Các phân tích trong đây phục vụ để có được insight và sử dụng chủ yếu cho Dashboard số 1. Không phải tất cả các thay đổi trong Notebooks này sẽ được sử dụng để train mô hình do vấn đề Data Leakage, chỉ một số các thay đổi được đánh giá là an toàn mới được sử dụng. Các thay đổi đó được xem xét thận trọng và fit chỉ trên tập train, perform trên tập val, test; sẽ được tiến hành sau quá trình train_test_split

Thay đổi bao gồm: 
1. Chuẩn hóa fullVisitorId và thời gian
2. Xác định revenue columns hiện có
3. Ép revenue về numeric
4. Kiểm tra missing, âm, outlier, zero/non-zero
5. Tính target chuẩn theo đề:
   target_log_revenue = log(1 + tổng transactionRevenue theo fullVisitorId)
6. Group theo fullVisitorId
7. In thống kê mô tả đầy đủ
8. So sánh target cũ nếu dataframe đã có target_log_revenue

| Cột                              | Đơn vị                       | Ý nghĩa                                                                                       |
| -------------------------------- | ---------------------------- | --------------------------------------------------------------------------------------------- |
| `transactionRevenue`             | **micro-dollar**             | Doanh thu gốc từ GA, phải chia `1_000_000` để ra dollar                                       |
| `totals_transactionRevenue`      | **micro-dollar**             | Giống `transactionRevenue`; bạn đã kiểm tra thấy 2 cột này khớp 100%                          |
| `totals_totalTransactionRevenue` | **micro-dollar**             | Tổng revenue/session theo GA, cũng ở dạng micro-dollar nhưng có thể khác `transactionRevenue` |
| `transactionRevenue_dollar`      | **dollar**                   | Đã được tạo bằng `transactionRevenue / 1_000_000`                                             |
| `revenue_raw`                    | **micro-dollar**             | Bản copy từ `transactionRevenue`                                                              |
| `revenue_dollar`                 | **dollar**                   | Bản copy từ `transactionRevenue_dollar`                                                       |
| `total_revenue_raw`              | **micro-dollar**             | Tổng `transactionRevenue` theo `fullVisitorId`                                                |
| `total_revenue_dollar`           | **dollar**                   | Tổng `transactionRevenue_dollar` theo `fullVisitorId`                                         |
| `target_log_revenue`             | **log của micro-dollar + 1** | `log1p(total_revenue_raw)` nếu tính đúng theo user                                            |
| `target_log_revenue_dollar`      | **log của dollar + 1**       | `log1p(total_revenue_dollar)`                                                                 |


# Hướng dẫn lấy dữ liệu cho Dashboard 1

1. Cohort Analysis

| Mục đích       | Cột nên dùng                          | Vì sao                                                                |
| -------------- | ------------------------------------- | --------------------------------------------------------------------- |
| ID khách hàng  | `fullVisitorId`                       | Gom session theo user                                                 |
| ID session     | `visitId`                             | Đếm số phiên                                                          |
| Ngày truy cập  | `date` hoặc `visitStartTime_datetime` | Xác định cohort month, return month                                   |
| Thứ tự phiên   | `visitNumber`                         | Biết user mới/quay lại                                                |
| Revenue raw    | `transactionRevenue`                  | Tính doanh thu theo cohort                                            |
| Revenue dollar | `transactionRevenue_dollar`           | Vẽ dashboard dễ hiểu                                                  |
| Target log     | `target_log_revenue`                  | Chỉ dùng nếu đã tính đúng theo user/window; không dùng cho cohort thô |


2. Cohort acquisition

| Mục đích      | Cột                          |
| ------------- | ---------------------------- |
| Kênh tổng     | `channelGrouping`            |
| Source        | `trafficSource_source`       |
| Medium        | `trafficSource_medium`       |
| Campaign      | `trafficSource_campaign`     |
| Keyword       | `trafficSource_keyword`      |
| Referral path | `trafficSource_referralPath` |
| True direct   | `trafficSource_isTrueDirect` |


3. Engagement

| Metric          | Cột                            |
| --------------- | ------------------------------ |
| Số phiên        | `visitId` hoặc `totals_visits` |
| Hits            | `totals_hits`                  |
| Pageviews       | `totals_pageviews`             |
| Bounce          | `totals_bounces`               |
| New visits      | `totals_newVisits`             |
| Time on site    | `totals_timeOnSite`            |
| Session quality | `totals_sessionQualityDim`     |


4. Geo/device dashboard

| Dashboard           | Cột                      |
| ------------------- | ------------------------ |
| Country performance | `geoNetwork_country`     |
| City performance    | `geoNetwork_city`        |
| Region              | `geoNetwork_region`      |
| Continent           | `geoNetwork_continent`   |
| Device category     | `device_deviceCategory`  |
| Browser             | `device_browser`         |
| Operating system    | `device_operatingSystem` |
| Mobile vs desktop   | `device_isMobile`        |


In [3]:
import pandas as pd
import numpy as np

# Load lại data từ file pickle
df_work = pd.read_pickle(r"D:\data_driven_marketing\df_full.pkl")

# Kiểm tra nhanh
print(df_work.shape)
df_work.head()
# 0. Copy data
# ============================================================

print("=" * 80)
print("0. BASIC DATA INFO")
print("=" * 80)

print("Shape:", df_work.shape)
print("Number of columns:", df_work.shape[1])
print("Columns:")
print(df_work.columns.tolist())

(1608337, 59)
0. BASIC DATA INFO
Shape: (1608337, 59)
Number of columns: 59
Columns:
['channelGrouping', 'date', 'fullVisitorId', 'visitId', 'visitNumber', 'visitStartTime', 'device_browser', 'device_operatingSystem', 'device_isMobile', 'device_deviceCategory', 'geoNetwork_continent', 'geoNetwork_subContinent', 'geoNetwork_country', 'geoNetwork_region', 'geoNetwork_metro', 'geoNetwork_city', 'geoNetwork_networkDomain', 'totals_visits', 'totals_hits', 'totals_pageviews', 'totals_bounces', 'totals_newVisits', 'totals_sessionQualityDim', 'totals_timeOnSite', 'totals_transactions', 'totals_transactionRevenue', 'totals_totalTransactionRevenue', 'trafficSource_campaign', 'trafficSource_source', 'trafficSource_medium', 'trafficSource_keyword', 'trafficSource_referralPath', 'trafficSource_isTrueDirect', 'trafficSource_adContent', 'trafficSource_adwordsClickInfo.page', 'trafficSource_adwordsClickInfo.slot', 'trafficSource_adwordsClickInfo.gclId', 'trafficSource_adwordsClickInfo.adNetworkType', 

In [4]:
# ============================================================
# 1. Required columns check
# ============================================================
print("\n" + "=" * 80)
print("1. REQUIRED COLUMNS CHECK")
print("=" * 80)

required_cols = ["fullVisitorId"]
missing_required = [c for c in required_cols if c not in df_work.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df_work["fullVisitorId"] = df_work["fullVisitorId"].astype(str)

print("fullVisitorId dtype:", df_work["fullVisitorId"].dtype)
print("Unique fullVisitorId:", df_work["fullVisitorId"].nunique())
print("Total rows:", len(df_work))


1. REQUIRED COLUMNS CHECK
fullVisitorId dtype: object
Unique fullVisitorId: 1252748
Total rows: 1608337


In [5]:
# ============================================================
# 2. Date / time normalization
# ============================================================
print("\n" + "=" * 80)
print("2. DATE / TIME CHECK")
print("=" * 80)

if "visitStartTime_datetime" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["visitStartTime_datetime"], errors="coerce")
elif "date" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["date"], errors="coerce")
elif "visitStartTime" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["visitStartTime"], unit="s", errors="coerce")
else:
    df_work["event_date"] = pd.NaT
    print("WARNING: No date column found.")

print("event_date missing:", df_work["event_date"].isna().sum())
print("Min date:", df_work["event_date"].min())
print("Max date:", df_work["event_date"].max())


2. DATE / TIME CHECK
event_date missing: 0
Min date: 2016-08-02 07:00:09
Max date: 2018-05-01 06:56:58


In [6]:
# ============================================================
# 3. Revenue columns detection
# ============================================================
print("\n" + "=" * 80)
print("3. REVENUE COLUMNS DETECTION")
print("=" * 80)

possible_revenue_cols = [
    "transactionRevenue",
    "transactionRevenue_dollar",
    "totals_transactionRevenue",
    "totals_totalTransactionRevenue",
    "target_log_revenue"
]

existing_revenue_cols = [c for c in possible_revenue_cols if c in df_work.columns]

print("Existing revenue-related columns:")
print(existing_revenue_cols)

if len(existing_revenue_cols) == 0:
    raise ValueError("No revenue-related columns found.")

for col in existing_revenue_cols:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

print("\nRevenue dtypes after numeric conversion:")
print(df_work[existing_revenue_cols].dtypes)



3. REVENUE COLUMNS DETECTION
Existing revenue-related columns:
['transactionRevenue', 'transactionRevenue_dollar', 'totals_transactionRevenue', 'totals_totalTransactionRevenue', 'target_log_revenue']

Revenue dtypes after numeric conversion:
transactionRevenue                float64
transactionRevenue_dollar         float64
totals_transactionRevenue         float64
totals_totalTransactionRevenue    float64
target_log_revenue                float64
dtype: object


In [7]:
# ============================================================
# 4. Choose canonical revenue source
# ============================================================
print("\n" + "=" * 80)
print("4. CHOOSE CANONICAL REVENUE SOURCE")
print("=" * 80)

if "transactionRevenue" in df_work.columns:
    revenue_col = "transactionRevenue"
elif "totals_transactionRevenue" in df_work.columns:
    revenue_col = "totals_transactionRevenue"
elif "totals_totalTransactionRevenue" in df_work.columns:
    revenue_col = "totals_totalTransactionRevenue"
else:
    raise ValueError(
        "No valid raw revenue column found. Only target_log_revenue exists, cannot reconstruct raw revenue safely."
    )

print("Canonical revenue column used:", revenue_col)

df_work[revenue_col] = df_work[revenue_col].fillna(0)
df_work["revenue_raw"] = df_work[revenue_col]

if "transactionRevenue_dollar" in df_work.columns:
    df_work["revenue_dollar"] = df_work["transactionRevenue_dollar"].fillna(0)
else:
    df_work["revenue_dollar"] = np.nan

print("Created columns: revenue_raw, revenue_dollar")



4. CHOOSE CANONICAL REVENUE SOURCE
Canonical revenue column used: transactionRevenue
Created columns: revenue_raw, revenue_dollar


In [8]:
# ============================================================
# 5. Revenue quality checks
# ============================================================
print("\n" + "=" * 80)
print("5. REVENUE QUALITY CHECKS")
print("=" * 80)

def revenue_quality_report(data, col):
    s = data[col]
    return {
        "column": col,
        "dtype": str(s.dtype),
        "n_rows": len(s),
        "n_missing": int(s.isna().sum()),
        "missing_rate": float(s.isna().mean()),
        "n_zero": int((s.fillna(0) == 0).sum()),
        "zero_rate": float((s.fillna(0) == 0).mean()),
        "n_positive": int((s.fillna(0) > 0).sum()),
        "positive_rate": float((s.fillna(0) > 0).mean()),
        "n_negative": int((s.fillna(0) < 0).sum()),
        "negative_rate": float((s.fillna(0) < 0).mean()),
        "min": float(s.min()) if s.notna().any() else np.nan,
        "max": float(s.max()) if s.notna().any() else np.nan,
        "mean": float(s.mean()) if s.notna().any() else np.nan,
        "median": float(s.median()) if s.notna().any() else np.nan,
        "std": float(s.std()) if s.notna().any() else np.nan,
        "n_unique": int(s.nunique(dropna=True))
    }

quality_df = pd.DataFrame([
    revenue_quality_report(df_work, col)
    for col in existing_revenue_cols
])

display(quality_df)

print("\nDescriptive statistics for revenue_raw:")
display(df_work["revenue_raw"].describe(percentiles=[
    0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99
]))

print("\nTop 20 largest revenue_raw rows:")
show_cols = ["fullVisitorId", "event_date", "revenue_raw"]
if "transactionRevenue_dollar" in df_work.columns:
    show_cols.append("transactionRevenue_dollar")

display(
    df_work.sort_values("revenue_raw", ascending=False)[show_cols].head(20)
)



5. REVENUE QUALITY CHECKS


,column,dtype,n_rows,n_missing,missing_rate,n_zero,zero_rate,n_positive,positive_rate,n_negative,negative_rate,min,max,mean,median,std,n_unique
0,transactionRevenue,float64,1608337,0,0.0,1590757,0.989069,17580,0.010931,0,0.0,0.0,2.312950e+10,1.371436e+06,0.0,4.597711e+07,7009
1,transactionRevenue_dollar,float64,1608337,0,0.0,1590757,0.989069,17580,0.010931,0,0.0,0.0,2.312950e+04,1.371436e+00,0.0,4.597711e+01,7009
2,totals_transactionRevenue,float64,1608337,0,0.0,1590757,0.989069,17580,0.010931,0,0.0,0.0,2.312950e+10,1.371436e+06,0.0,4.597711e+07,7009
3,totals_totalTransactionRevenue,float64,1608337,0,0.0,1590757,0.989069,17580,0.010931,0,0.0,0.0,4.708206e+10,1.568682e+06,0.0,7.046884e+07,8220
4,target_log_revenue,float64,1608337,0,0.0,1590757,0.989069,17580,0.010931,0,0.0,0.0,2.386437e+01,1.943090e-01,0.0,1.852491e+00,7009



Descriptive statistics for revenue_raw:


count    1.608337e+06
mean     1.371436e+06
std      4.597711e+07
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
90%      0.000000e+00
95%      0.000000e+00
99%      1.329000e+07
max      2.312950e+10
Name: revenue_raw, dtype: float64


Top 20 largest revenue_raw rows:


,fullVisitorId,event_date,revenue_raw,transactionRevenue_dollar
1361871,1957458976293878100,2017-04-05 20:19:40,2.312950e+10,23129.50
1472897,1957458976293878100,2017-02-14 18:30:28,1.785550e+10,17855.50
1376484,5632276788326171571,2016-09-16 14:20:43,1.602375e+10,16023.75
220100,1957458976293878100,2017-09-13 13:56:28,1.322940e+10,13229.40
1518403,1957458976293878100,2017-08-23 15:12:37,1.229300e+10,12293.00
649340,9417857471295131045,2017-07-18 19:00:09,1.058914e+10,10589.14
516657,1957458976293878100,2017-08-02 17:20:28,9.925110e+09,9925.11
893239,1957458976293878100,2017-03-24 18:36:00,8.677830e+09,8677.83
549475,4471415710206918415,2017-04-27 13:31:56,8.248800e+09,8248.80
979065,2050125810100188362,2017-09-01 19:23:06,7.427430e+09,7427.43


In [9]:
# ============================================================
# 6. Revenue column consistency
# ============================================================
print("\n" + "=" * 80)
print("6. REVENUE COLUMN CONSISTENCY CHECK")
print("=" * 80)

if {"transactionRevenue", "totals_transactionRevenue"}.issubset(df_work.columns):
    tmp = df_work[["transactionRevenue", "totals_transactionRevenue"]].fillna(0).copy()
    diff = (tmp["transactionRevenue"] - tmp["totals_transactionRevenue"]).abs()

    print("transactionRevenue vs totals_transactionRevenue")
    print("Rows different:", int((diff > 0).sum()))
    print("Different rate:", float((diff > 0).mean()))
    print("Max absolute diff:", diff.max())

    if (diff > 0).sum() > 0:
        print("\nSample different rows:")
        display(
            df_work.loc[diff > 0, [
                "fullVisitorId",
                "event_date",
                "transactionRevenue",
                "totals_transactionRevenue"
            ]].head(20)
        )

if "transactionRevenue_dollar" in df_work.columns:
    dollar_positive = df_work["transactionRevenue_dollar"].fillna(0) > 0
    raw_positive = df_work["revenue_raw"].fillna(0) > 0

    print("\nRaw revenue positive rows:", int(raw_positive.sum()))
    print("Dollar revenue positive rows:", int(dollar_positive.sum()))
    print("Rows where raw positive but dollar zero:", int((raw_positive & ~dollar_positive).sum()))
    print("Rows where dollar positive but raw zero:", int((dollar_positive & ~raw_positive).sum()))

    if raw_positive.sum() > 0 and dollar_positive.sum() > 0:
        ratio = (
            df_work.loc[raw_positive & dollar_positive, "revenue_raw"] /
            df_work.loc[raw_positive & dollar_positive, "transactionRevenue_dollar"]
        )
        print("\nRaw / dollar ratio description:")
        display(ratio.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))




6. REVENUE COLUMN CONSISTENCY CHECK
transactionRevenue vs totals_transactionRevenue
Rows different: 0
Different rate: 0.0
Max absolute diff: 0.0

Raw revenue positive rows: 17580
Dollar revenue positive rows: 17580
Rows where raw positive but dollar zero: 0
Rows where dollar positive but raw zero: 0

Raw / dollar ratio description:


count    1.758000e+04
mean     1.000000e+06
std      6.377720e-11
min      1.000000e+06
1%       1.000000e+06
5%       1.000000e+06
50%      1.000000e+06
95%      1.000000e+06
99%      1.000000e+06
max      1.000000e+06
dtype: float64

In [10]:
# ============================================================
# 7. Create official user-level target
# ============================================================
print("\n" + "=" * 80)
print("7. CREATE USER-LEVEL TARGET")
print("=" * 80)

agg_dict = {
    "n_rows": ("fullVisitorId", "size"),
    "first_date": ("event_date", "min"),
    "last_date": ("event_date", "max"),
    "total_revenue_raw": ("revenue_raw", "sum"),
    "max_session_revenue_raw": ("revenue_raw", "max"),
    "mean_session_revenue_raw": ("revenue_raw", "mean"),
    "n_revenue_sessions": ("revenue_raw", lambda x: int((x > 0).sum()))
}

if "visitId" in df_work.columns:
    agg_dict["n_sessions"] = ("visitId", "nunique")
else:
    agg_dict["n_sessions"] = ("fullVisitorId", "size")

user_target = (
    df_work
    .groupby("fullVisitorId", as_index=False)
    .agg(**agg_dict)
)

user_target["target_log_revenue"] = np.log1p(user_target["total_revenue_raw"])
user_target["is_buyer"] = (user_target["total_revenue_raw"] > 0).astype(int)

if "transactionRevenue_dollar" in df_work.columns:
    user_dollar = (
        df_work
        .groupby("fullVisitorId", as_index=False)
        .agg(
            total_revenue_dollar=(
                "transactionRevenue_dollar",
                lambda x: pd.to_numeric(x, errors="coerce").fillna(0).sum()
            )
        )
    )
    user_target = user_target.merge(user_dollar, on="fullVisitorId", how="left")
    user_target["target_log_revenue_dollar"] = np.log1p(user_target["total_revenue_dollar"])
else:
    user_target["total_revenue_dollar"] = np.nan
    user_target["target_log_revenue_dollar"] = np.nan

print("User-level target shape:", user_target.shape)
display(user_target.head())



7. CREATE USER-LEVEL TARGET
User-level target shape: (1252748, 13)


,fullVisitorId,n_rows,first_date,last_date,total_revenue_raw,max_session_revenue_raw,mean_session_revenue_raw,n_revenue_sessions,n_sessions,target_log_revenue,is_buyer,total_revenue_dollar,target_log_revenue_dollar
0,0000000259678714014,2,2017-11-28 23:33:21,2017-11-29 00:19:40,0.0,0.0,0.0,0,2,0.0,0,0.0,0.0
1,0000010278554503158,1,2016-10-21 05:57:46,2016-10-21 05:57:46,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
2,0000020424342248747,1,2016-12-01 07:55:01,2016-12-01 07:55:01,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
3,0000027376579751715,1,2017-02-12 02:24:53,2017-02-12 02:24:53,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
4,0000039460501403861,1,2017-03-27 15:45:16,2017-03-27 15:45:16,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0


In [11]:
# ============================================================
# 8. User-level target descriptive statistics
# ============================================================
print("\n" + "=" * 80)
print("8. USER-LEVEL TARGET DESCRIPTIVE STATISTICS")
print("=" * 80)

print("Total users:", user_target["fullVisitorId"].nunique())
print("Total rows:", len(df_work))
print("Buyer users:", int(user_target["is_buyer"].sum()))
print("Buyer user rate:", user_target["is_buyer"].mean())
print("Zero-revenue users:", int((user_target["total_revenue_raw"] == 0).sum()))
print("Zero-revenue user rate:", (user_target["total_revenue_raw"] == 0).mean())

print("\nUser-level total_revenue_raw describe:")
display(user_target["total_revenue_raw"].describe(percentiles=[
    0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99
]))

print("\nUser-level target_log_revenue describe:")
display(user_target["target_log_revenue"].describe(percentiles=[
    0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99
]))

print("\nBuyer-only total_revenue_raw describe:")
display(
    user_target.loc[user_target["is_buyer"] == 1, "total_revenue_raw"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nBuyer-only target_log_revenue describe:")
display(
    user_target.loc[user_target["is_buyer"] == 1, "target_log_revenue"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)


8. USER-LEVEL TARGET DESCRIPTIVE STATISTICS
Total users: 1252748
Total rows: 1608337
Buyer users: 15391
Buyer user rate: 0.012285790917247522
Zero-revenue users: 1237357
Zero-revenue user rate: 0.9877142090827524

User-level total_revenue_raw describe:


count    1.252748e+06
mean     1.760714e+06
std      1.174416e+08
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
90%      0.000000e+00
95%      0.000000e+00
99%      1.919000e+07
max      1.192475e+11
Name: total_revenue_raw, dtype: float64


User-level target_log_revenue describe:


count    1.252748e+06
mean     2.185937e-01
std      1.964565e+00
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
90%      0.000000e+00
95%      0.000000e+00
99%      1.676990e+01
max      2.550447e+01
Name: target_log_revenue, dtype: float64


Buyer-only total_revenue_raw describe:


count    1.539100e+04
mean     1.433130e+08
std      1.049965e+09
min      1.000000e+04
1%       3.400000e+06
5%       9.975000e+06
10%      1.399000e+07
25%      2.349000e+07
50%      4.718000e+07
75%      1.022250e+08
90%      2.589300e+08
95%      4.910450e+08
99%      1.500075e+09
max      1.192475e+11
Name: total_revenue_raw, dtype: float64


Buyer-only target_log_revenue describe:


count    15391.000000
mean        17.792401
std          1.210038
min          9.210440
1%          15.039286
5%          16.115592
10%         16.453853
25%         16.972085
50%         17.669481
75%         18.442687
90%         19.372068
95%         20.012046
99%         21.128779
max         25.504467
Name: target_log_revenue, dtype: float64

In [12]:
# ============================================================
# 9. Skewness / kurtosis
# ============================================================
print("\n" + "=" * 80)
print("9. SKEWNESS / KURTOSIS")
print("=" * 80)

print("Session-level revenue_raw skew:", df_work["revenue_raw"].skew())
print("Session-level revenue_raw kurtosis:", df_work["revenue_raw"].kurtosis())

print("User-level total_revenue_raw skew:", user_target["total_revenue_raw"].skew())
print("User-level total_revenue_raw kurtosis:", user_target["total_revenue_raw"].kurtosis())

print("User-level target_log_revenue skew:", user_target["target_log_revenue"].skew())
print("User-level target_log_revenue kurtosis:", user_target["target_log_revenue"].kurtosis())



9. SKEWNESS / KURTOSIS
Session-level revenue_raw skew: 219.07750178462157
Session-level revenue_raw kurtosis: 78057.65620773396
User-level total_revenue_raw skew: 845.1142404431489
User-level total_revenue_raw kurtosis: 849277.0749344262
User-level target_log_revenue skew: 8.919328239714114
User-level target_log_revenue kurtosis: 77.96050386848437


In [13]:
# ============================================================
# 10. Compare existing target_log_revenue if available
# ============================================================
print("\n" + "=" * 80)
print("10. COMPARE WITH EXISTING target_log_revenue COLUMN")
print("=" * 80)

if "target_log_revenue" in df.columns:
    existing_target = df_work[["fullVisitorId", "target_log_revenue"]].copy()
    existing_target["target_log_revenue"] = pd.to_numeric(
        existing_target["target_log_revenue"],
        errors="coerce"
    )

    existing_user_target = (
        existing_target
        .groupby("fullVisitorId", as_index=False)
        .agg(
            existing_target_max=("target_log_revenue", "max"),
            existing_target_mean=("target_log_revenue", "mean"),
            existing_target_sum=("target_log_revenue", "sum"),
            existing_target_nunique=("target_log_revenue", "nunique")
        )
    )

    compare_target = user_target.merge(existing_user_target, on="fullVisitorId", how="left")

    compare_target["diff_recomputed_vs_existing_max"] = (
        compare_target["target_log_revenue"] -
        compare_target["existing_target_max"]
    )

    display(
        compare_target[[
            "fullVisitorId",
            "target_log_revenue",
            "existing_target_max",
            "existing_target_mean",
            "existing_target_sum",
            "existing_target_nunique",
            "diff_recomputed_vs_existing_max"
        ]].head()
    )

    print("\nDifference describe:")
    display(compare_target["diff_recomputed_vs_existing_max"].describe())

    print("\nUsers with abs(diff) > 1e-6:")
    print(int((compare_target["diff_recomputed_vs_existing_max"].abs() > 1e-6).sum()), "out of", len(compare_target))
else:
    print("No existing target_log_revenue column found.")


10. COMPARE WITH EXISTING target_log_revenue COLUMN


,fullVisitorId,target_log_revenue,existing_target_max,existing_target_mean,existing_target_sum,existing_target_nunique,diff_recomputed_vs_existing_max
0,0000000259678714014,0.0,0.0,0.0,0.0,1,0.0
1,0000010278554503158,0.0,0.0,0.0,0.0,1,0.0
2,0000020424342248747,0.0,0.0,0.0,0.0,1,0.0
3,0000027376579751715,0.0,0.0,0.0,0.0,1,0.0
4,0000039460501403861,0.0,0.0,0.0,0.0,1,0.0



Difference describe:


count    1.252748e+06
mean     5.900043e-04
std      1.996424e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.989659e+00
Name: diff_recomputed_vs_existing_max, dtype: float64


Users with abs(diff) > 1e-6:
1436 out of 1252748


In [14]:
# ============================================================
# 11. Potential issue flags
# ============================================================
print("\n" + "=" * 80)
print("11. POTENTIAL ISSUE FLAGS")
print("=" * 80)

issues = []

if df_work["revenue_raw"].isna().sum() > 0:
    issues.append("revenue_raw still has missing values.")

if (df_work["revenue_raw"] < 0).sum() > 0:
    issues.append("Negative revenue values exist.")

if user_target["is_buyer"].mean() < 0.01:
    issues.append("Buyer rate is very low (<1%). Expect severe zero-inflation.")

if user_target["target_log_revenue"].max() > 30:
    issues.append("Very large log target detected. Check whether revenue is stored in micros.")

if "transactionRevenue_dollar" in df_work.columns and df_work["transactionRevenue_dollar"].max() > 0:
    raw_dollar_ratio = (
        df_work.loc[
            (df_work["revenue_raw"] > 0) &
            (df_work["transactionRevenue_dollar"] > 0),
            "revenue_raw"
        ] /
        df_work.loc[
            (df_work["revenue_raw"] > 0) &
            (df_work["transactionRevenue_dollar"] > 0),
            "transactionRevenue_dollar"
        ]
    )

    if len(raw_dollar_ratio) > 0:
        median_ratio = raw_dollar_ratio.median()
        if median_ratio > 100000:
            issues.append(
                f"Raw revenue seems much larger than dollar revenue. Median raw/dollar ratio = {median_ratio:.2f}. "
                "This may indicate raw revenue is stored in micros."
            )

if len(issues) == 0:
    print("No obvious critical issues found.")
else:
    print("Issues / warnings:")
    for i, issue in enumerate(issues, 1):
        print(f"{i}. {issue}")



11. POTENTIAL ISSUE FLAGS
Issues / warnings:
1. Raw revenue seems much larger than dollar revenue. Median raw/dollar ratio = 1000000.00. This may indicate raw revenue is stored in micros.


In [15]:
# ============================================================
# 12. Final outputs
# ============================================================
print("\n" + "=" * 80)
print("12. FINAL OUTPUTS")
print("=" * 80)

print("Input dataframe: df")
print("Working dataframe created: df_work")
print("User-level target dataframe created: user_target")
print("Recommended modeling target column: target_log_revenue")
print("Recommended raw user revenue column: total_revenue_raw")
print("Recommended buyer classification column: is_buyer")

display(user_target.head())

# Optional save
user_target.to_pickle("user_level_target_check.pkl")
user_target.to_csv("user_level_target_check.csv", index=False)


12. FINAL OUTPUTS
Input dataframe: df
Working dataframe created: df_work
User-level target dataframe created: user_target
Recommended modeling target column: target_log_revenue
Recommended raw user revenue column: total_revenue_raw
Recommended buyer classification column: is_buyer


,fullVisitorId,n_rows,first_date,last_date,total_revenue_raw,max_session_revenue_raw,mean_session_revenue_raw,n_revenue_sessions,n_sessions,target_log_revenue,is_buyer,total_revenue_dollar,target_log_revenue_dollar
0,0000000259678714014,2,2017-11-28 23:33:21,2017-11-29 00:19:40,0.0,0.0,0.0,0,2,0.0,0,0.0,0.0
1,0000010278554503158,1,2016-10-21 05:57:46,2016-10-21 05:57:46,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
2,0000020424342248747,1,2016-12-01 07:55:01,2016-12-01 07:55:01,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
3,0000027376579751715,1,2017-02-12 02:24:53,2017-02-12 02:24:53,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0
4,0000039460501403861,1,2017-03-27 15:45:16,2017-03-27 15:45:16,0.0,0.0,0.0,0,1,0.0,0,0.0,0.0


In [16]:
import json
from pathlib import Path

# =========================
# 1. Đường dẫn notebook hiện tại
# =========================
# Sửa tên file .ipynb của bạn tại đây
notebook_path = Path("Target_col_analysis.ipynb")

# File output txt
output_txt_path = Path("all_notebook_outputs.txt")

# =========================
# 2. Đọc file ipynb
# =========================
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

all_outputs = []

# =========================
# 3. Duyệt từng cell và lấy output
# =========================
for i, cell in enumerate(nb.get("cells", []), start=1):
    cell_type = cell.get("cell_type", "")

    if cell_type != "code":
        continue

    outputs = cell.get("outputs", [])

    if not outputs:
        continue

    all_outputs.append("=" * 100)
    all_outputs.append(f"CELL {i}")
    all_outputs.append("=" * 100)

    source_code = "".join(cell.get("source", []))
    all_outputs.append("\n[CODE]\n")
    all_outputs.append(source_code)

    all_outputs.append("\n[OUTPUT]\n")

    for output in outputs:
        output_type = output.get("output_type", "")

        # print(), stdout, stderr
        if output_type == "stream":
            text = output.get("text", "")
            if isinstance(text, list):
                text = "".join(text)
            all_outputs.append(text)

        # errors / traceback
        elif output_type == "error":
            ename = output.get("ename", "")
            evalue = output.get("evalue", "")
            traceback = output.get("traceback", [])

            all_outputs.append(f"{ename}: {evalue}\n")
            all_outputs.append("\n".join(traceback))

        # display(), dataframe text/plain, plots metadata, etc.
        elif output_type in ["execute_result", "display_data"]:
            data = output.get("data", {})

            if "text/plain" in data:
                text = data["text/plain"]
                if isinstance(text, list):
                    text = "".join(text)
                all_outputs.append(text)

            elif "text/html" in data:
                html = data["text/html"]
                if isinstance(html, list):
                    html = "".join(html)
                all_outputs.append("[HTML OUTPUT]\n")
                all_outputs.append(html)

            elif "image/png" in data:
                all_outputs.append("[IMAGE OUTPUT: image/png omitted]\n")

            else:
                all_outputs.append(f"[UNSUPPORTED DISPLAY DATA KEYS: {list(data.keys())}]\n")

    all_outputs.append("\n\n")

# =========================
# 4. Ghi ra file txt
# =========================
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(all_outputs))

print(f"Done. Saved notebook outputs to: {output_txt_path.resolve()}")

Done. Saved notebook outputs to: D:\data_driven_marketing\Notebooks\all_notebook_outputs.txt
